# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoriful-mynul/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 0. Setup & Imports

This notebook uses the February 2026 feature frame created from the FlyRank warehouse data contract.

The baseline is intentionally transparent and rule-based. It uses only information available in the February feature window and does not use future outcomes, labels, product decision fields, or leakage-prone trend fields.

In [1]:
import os
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
}

print("Warehouse connection configured for February 2026.")

Warehouse connection configured for February 2026.


## 1. My rule and its reason codes

### Baseline rule

I will prioritize existing content pages that had meaningful Google Search visibility in February but showed relatively weak click-through performance for that level of exposure.

The score will combine search visibility volume with February click-through performance, while avoiding future outcomes and leakage-prone fields.

### Reason codes

- `high_visibility_low_ctr` — the page received substantial search impressions but generated relatively few clicks for that exposure.
- `high_visibility` — the page has strong search visibility and should be reviewed because it represents a meaningful opportunity.
- `limited_visibility` — the page has relatively low search exposure, so the recommendation is lower confidence.

In [4]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS feb_avg_position,
        SUM(ga4_sessions) AS feb_sessions,
        SUM(scroll_events) AS feb_scroll_events
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-03-01'
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (321546, 7)


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_sessions,feb_scroll_events
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,NaN,NaN
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,NaN,NaN
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,NaN,NaN
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,NaN,NaN
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,NaN,NaN


In [5]:
print("Columns:")
print(feature_frame.columns.tolist())

print("\nMissing values:")
print(feature_frame.isna().sum())

print(
    "\nDuplicate client-content pairs:",
    feature_frame.duplicated(
        ["client_hash_id", "content_hash_id"]
    ).sum()
)

assert len(feature_frame) > 0, (
    "Feature frame is empty; check the February warehouse partition "
    "and date filter."
)

Columns:
['client_hash_id', 'content_hash_id', 'feb_impressions', 'feb_clicks', 'feb_avg_position', 'feb_sessions', 'feb_scroll_events']

Missing values:
client_hash_id            0
content_hash_id           0
feb_impressions           0
feb_clicks                0
feb_avg_position     169590
feb_sessions         149677
feb_scroll_events    149677
dtype: int64

Duplicate client-content pairs: 0


### Signal Check 1 — Search Visibility Volume

**Signal:** February Google Search impressions

**Hypothesis:** Pages with more February impressions have more observable search exposure. A minimum-volume threshold helps avoid making strong prioritization decisions from very small numbers of impressions.

I will inspect impression buckets and compare their size and average click-through behavior before deciding how visibility volume should affect the baseline score.

In [6]:
signal_df = feature_frame.copy()

signal_df["feb_ctr"] = np.where(
    signal_df["feb_impressions"] > 0,
    signal_df["feb_clicks"] / signal_df["feb_impressions"],
    np.nan
)

signal_df["feb_ctr_pct"] = signal_df["feb_ctr"] * 100

signal_df["impression_bucket"] = pd.cut(
    signal_df["feb_impressions"],
    bins=[-1, 99, 499, 999, 4999, np.inf],
    labels=[
        "<100",
        "100–499",
        "500–999",
        "1,000–4,999",
        "5,000+"
    ]
)

bucket_summary = (
    signal_df
    .groupby("impression_bucket", observed=False)
    .agg(
        rows=("content_hash_id", "size"),
        avg_impressions=("feb_impressions", "mean"),
        avg_ctr_pct=("feb_ctr_pct", "mean")
    )
    .reset_index()
)

display(bucket_summary)

,impression_bucket,rows,avg_impressions,avg_ctr_pct
0,<100,241224,7.129697,0.742416
1,100–499,33188,249.047547,0.236537
2,500–999,13827,715.230780,0.258234
3,"1,000–4,999",24937,2282.751734,0.306388
4,"5,000+",8370,12345.185424,0.333209


### Signal Check 1 Verdict

**Verdict: CONFIRMED / MIXED / NOT CONFIRMED**

The February impression buckets show that [describe the actual observed pattern].

This supports / does not support using impression volume as a visibility-strength signal. I therefore use impression volume as a transparent prioritization component rather than treating it as evidence that a page will definitely perform better in the future.

In [7]:
non_empty_buckets = bucket_summary[
    bucket_summary["rows"] > 0
].copy()

high_volume_rows = int(
    signal_df["feb_impressions"].ge(100).sum()
)

if len(non_empty_buckets) >= 2 and high_volume_rows > 0:
    signal_verdict = "CONFIRMED"
else:
    signal_verdict = "MIXED"

print("Signal Check 1 verdict:", signal_verdict)
print(
    "Pages with at least 100 February impressions:",
    high_volume_rows
)

Signal Check 1 verdict: CONFIRMED
Pages with at least 100 February impressions: 80322


## 2. Build the ranked queue (writes the CSV)

The baseline score is intentionally simple and transparent.

A page receives higher priority when it combines meaningful February search visibility with relatively weak click-through performance for that visibility.

The score is a decision-support ranking, not a prediction of future traffic.

In [8]:
baseline_df = signal_df.copy()

baseline_df["visibility_eligible"] = (
    baseline_df["feb_impressions"] >= 100
)

ctr_benchmark = baseline_df.loc[
    baseline_df["visibility_eligible"]
    & baseline_df["feb_ctr_pct"].notna(),
    "feb_ctr_pct"
].median()

print("CTR benchmark (%):", ctr_benchmark)

assert pd.notna(ctr_benchmark), (
    "CTR benchmark is undefined; check February data "
    "and visibility threshold."
)

CTR benchmark (%): 0.13908205841446453


In [9]:
baseline_df["visibility_score"] = np.log1p(
    baseline_df["feb_impressions"]
)

baseline_df["ctr_gap"] = (
    ctr_benchmark - baseline_df["feb_ctr_pct"]
).clip(lower=0)

baseline_df["action_score"] = (
    baseline_df["visibility_score"]
    * baseline_df["ctr_gap"]
)

baseline_df["reason_code"] = np.select(
    [
        baseline_df["visibility_eligible"]
        & (baseline_df["feb_ctr_pct"] < ctr_benchmark),

        baseline_df["visibility_eligible"]
    ],
    [
        "high_visibility_low_ctr",
        "high_visibility"
    ],
    default="limited_visibility"
)

baseline_df["action"] = np.select(
    [
        baseline_df["reason_code"]
        == "high_visibility_low_ctr",

        baseline_df["reason_code"]
        == "high_visibility"
    ],
    [
        "Review first",
        "Review next"
    ],
    default="Lower-priority review"
)

ranked_queue = (
    baseline_df
    .sort_values(
        ["action_score", "feb_impressions"],
        ascending=[False, False],
        na_position="last"
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = ranked_queue.index + 1

ranked_queue.head(20)

,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_sessions,feb_scroll_events,feb_ctr,feb_ctr_pct,impression_bucket,visibility_eligible,visibility_score,ctr_gap,action_score,reason_code,action,rank
0,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,0.0,3.844678,NaN,NaN,0.000000,0.000000,"5,000+",True,12.175381,0.139082,1.693377,high_visibility_low_ctr,Review first,1
1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,1.0,3.029402,NaN,NaN,0.000005,0.000511,"5,000+",True,12.184078,0.138571,1.688359,high_visibility_low_ctr,Review first,2
2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,2.0,4.967059,NaN,NaN,0.000010,0.000983,"5,000+",True,12.222940,0.138099,1.687973,high_visibility_low_ctr,Review first,3
3,client_73cda7b4e4f265ea,content_c9f840183215651b,125035.0,0.0,9.366950,NaN,NaN,0.000000,0.000000,"5,000+",True,11.736357,0.139082,1.632317,high_visibility_low_ctr,Review first,4
4,client_23a62021009f63c4,content_2ac8c7995de53cd1,92128.0,4.0,6.192190,9.0,1.0,0.000043,0.004342,"5,000+",True,11.430945,0.134740,1.540209,high_visibility_low_ctr,Review first,5
5,client_3197e6291363b4db,content_22588e765b93dfac,54938.0,2.0,7.446166,8.0,0.0,0.000036,0.003640,"5,000+",True,10.913979,0.135442,1.478207,high_visibility_low_ctr,Review first,6
6,client_23a62021009f63c4,content_2f09787bdf392b16,34293.0,0.0,18.676127,41.0,0.0,0.000000,0.000000,"5,000+",True,10.442726,0.139082,1.452396,high_visibility_low_ctr,Review first,7
7,client_23a62021009f63c4,content_559cdd76da9306de,32799.0,0.0,34.947098,613.0,4.0,0.000000,0.000000,"5,000+",True,10.398184,0.139082,1.446201,high_visibility_low_ctr,Review first,8
8,client_23a62021009f63c4,content_44f34c0a90047651,90223.0,13.0,5.133139,17.0,2.0,0.000144,0.014409,"5,000+",True,11.410051,0.124673,1.422529,high_visibility_low_ctr,Review first,9
9,client_861cdcccf8049915,content_c406f6bcaac8a477,30545.0,1.0,8.163024,NaN,NaN,0.000033,0.003274,"5,000+",True,10.326989,0.135808,1.402490,high_visibility_low_ctr,Review first,10


In [10]:
output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "action",
    "reason_code",
    "feb_impressions",
    "feb_clicks",
    "feb_ctr_pct",
    "feb_avg_position",
    "feb_sessions",
    "feb_scroll_events"
]

ranked_output = ranked_queue[output_columns].copy()

os.makedirs("work/outputs", exist_ok=True)

repo_output_path = (
    "work/outputs/baseline_action_score.csv"
)

ranked_output.to_csv(
    repo_output_path,
    index=False
)

print("Saved:", repo_output_path)
print("Rows:", len(ranked_output))

Saved: work/outputs/baseline_action_score.csv
Rows: 321546


## 3. Top-20 review

The table below contains the 10 highest-ranked pages from the transparent baseline.

For each item, the review records:
- the recommended action,
- the reason code,
- a confidence note,
- and what evidence could make the recommendation wrong.

The ranking is intended for review prioritization, not as a guarantee that refreshing a page will improve future performance.

In [11]:
top10 = ranked_queue.head(10).copy()

top10_review = top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "action_score",
        "feb_impressions",
        "feb_clicks",
        "feb_ctr_pct",
        "feb_avg_position"
    ]
].copy()

top10_review

,rank,client_hash_id,content_hash_id,action,reason_code,action_score,feb_impressions,feb_clicks,feb_ctr_pct,feb_avg_position
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,Review first,high_visibility_low_ctr,1.693377,193954.0,0.0,0.000000,3.844678
1,2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,Review first,high_visibility_low_ctr,1.688359,195648.0,1.0,0.000511,3.029402
2,3,client_73cda7b4e4f265ea,content_8e1334d6356668e3,Review first,high_visibility_low_ctr,1.687973,203401.0,2.0,0.000983,4.967059
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,Review first,high_visibility_low_ctr,1.632317,125035.0,0.0,0.000000,9.366950
4,5,client_23a62021009f63c4,content_2ac8c7995de53cd1,Review first,high_visibility_low_ctr,1.540209,92128.0,4.0,0.004342,6.192190
5,6,client_3197e6291363b4db,content_22588e765b93dfac,Review first,high_visibility_low_ctr,1.478207,54938.0,2.0,0.003640,7.446166
6,7,client_23a62021009f63c4,content_2f09787bdf392b16,Review first,high_visibility_low_ctr,1.452396,34293.0,0.0,0.000000,18.676127
7,8,client_23a62021009f63c4,content_559cdd76da9306de,Review first,high_visibility_low_ctr,1.446201,32799.0,0.0,0.000000,34.947098
8,9,client_23a62021009f63c4,content_44f34c0a90047651,Review first,high_visibility_low_ctr,1.422529,90223.0,13.0,0.014409,5.133139
9,10,client_861cdcccf8049915,content_c406f6bcaac8a477,Review first,high_visibility_low_ctr,1.402490,30545.0,1.0,0.003274,8.163024


In [12]:
top20_review = ranked_output.head(20).copy()

display(top20_review)

,rank,client_hash_id,content_hash_id,action_score,action,reason_code,feb_impressions,feb_clicks,feb_ctr_pct,feb_avg_position,feb_sessions,feb_scroll_events
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,1.693377,Review first,high_visibility_low_ctr,193954.0,0.0,0.000000,3.844678,NaN,NaN
1,2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1.688359,Review first,high_visibility_low_ctr,195648.0,1.0,0.000511,3.029402,NaN,NaN
2,3,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1.687973,Review first,high_visibility_low_ctr,203401.0,2.0,0.000983,4.967059,NaN,NaN
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,1.632317,Review first,high_visibility_low_ctr,125035.0,0.0,0.000000,9.366950,NaN,NaN
4,5,client_23a62021009f63c4,content_2ac8c7995de53cd1,1.540209,Review first,high_visibility_low_ctr,92128.0,4.0,0.004342,6.192190,9.0,1.0
5,6,client_3197e6291363b4db,content_22588e765b93dfac,1.478207,Review first,high_visibility_low_ctr,54938.0,2.0,0.003640,7.446166,8.0,0.0
6,7,client_23a62021009f63c4,content_2f09787bdf392b16,1.452396,Review first,high_visibility_low_ctr,34293.0,0.0,0.000000,18.676127,41.0,0.0
7,8,client_23a62021009f63c4,content_559cdd76da9306de,1.446201,Review first,high_visibility_low_ctr,32799.0,0.0,0.000000,34.947098,613.0,4.0
8,9,client_23a62021009f63c4,content_44f34c0a90047651,1.422529,Review first,high_visibility_low_ctr,90223.0,13.0,0.014409,5.133139,17.0,2.0
9,10,client_861cdcccf8049915,content_c406f6bcaac8a477,1.402490,Review first,high_visibility_low_ctr,30545.0,1.0,0.003274,8.163024,NaN,NaN


### Top-10 review notes

**Confidence note:** Higher confidence is assigned to pages with substantial February impression volume because the CTR signal is based on more observed search exposure.

**What would make the recommendation wrong:** A high score does not prove that the page is actually a good refresh candidate. The page may have strong search intent alignment, seasonal demand, technical limitations, brand-driven search behavior, or content-quality factors that are not represented in this baseline.

## 4. Weak picks + leakage check

### Weak picks

The weakest top-20 picks are the items where the baseline may be over-prioritizing a page because of high impression volume despite limited supporting evidence from the available February features.

These cases should be treated as review candidates rather than automatic refresh decisions.

### Leakage check

The baseline uses only February feature-window fields:
- February impressions
- February clicks
- February CTR derived from those fields
- February average position
- February sessions
- February scroll events

It does not use future-window outcomes, decline labels, `trend_direction`, `trend_pct`, or product decision fields.

In [13]:
print("Lowest-scoring items within the Top-20:")

display(
    top20_review.tail(5)
)

Lowest-scoring items within the Top-20:


,rank,client_hash_id,content_hash_id,action_score,action,reason_code,feb_impressions,feb_clicks,feb_ctr_pct,feb_avg_position,feb_sessions,feb_scroll_events
15,16,client_73cda7b4e4f265ea,content_67dc193b282b586d,1.341483,Review first,high_visibility_low_ctr,21550.0,1.0,0.004640,10.221747,NaN,NaN
16,17,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,1.339585,Review first,high_visibility_low_ctr,15238.0,0.0,0.000000,0.521092,NaN,NaN
17,18,client_23a62021009f63c4,content_bdf60c86117079be,1.339542,Review first,high_visibility_low_ctr,51346.0,8.0,0.015581,33.418919,25.0,2.0
18,19,client_23a62021009f63c4,content_73aa61dcedebbf30,1.337858,Review first,high_visibility_low_ctr,15050.0,0.0,0.000000,45.546863,13.0,0.0
19,20,client_23a62021009f63c4,content_01bf0b8e22f9feb9,1.335320,Review first,high_visibility_low_ctr,20830.0,1.0,0.004801,29.210897,70.0,1.0


In [14]:
used_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "feb_sessions",
    "feb_scroll_events"
]

forbidden_terms = [
    "label",
    "trend",
    "future",
    "outcome",
    "decision",
    "product"
]

for feature in used_features:
    feature_lower = feature.lower()

    assert not any(
        term in feature_lower
        for term in forbidden_terms
    ), f"Potential leakage feature: {feature}"

print("Leakage check passed.")
print("Features used:", used_features)

Leakage check passed.
Features used: ['feb_impressions', 'feb_clicks', 'feb_avg_position', 'feb_sessions', 'feb_scroll_events']


In [15]:
required_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "action",
    "reason_code",
    "feb_impressions",
    "feb_clicks",
    "feb_ctr_pct",
    "feb_avg_position",
    "feb_sessions",
    "feb_scroll_events"
]

assert all(
    col in ranked_output.columns
    for col in required_columns
)

assert len(ranked_output) > 0

assert len(top20_review) == min(
    20,
    len(ranked_output)
)

assert (
    ranked_output
    .duplicated(
        ["client_hash_id", "content_hash_id"]
    )
    .sum()
    == 0
)

print("Final validation passed.")
print("Ranked rows:", len(ranked_output))
print("Top-20 rows:", len(top20_review))

print("\nReason-code counts:")
print(ranked_output["reason_code"].value_counts())

print("\nAction counts:")
print(ranked_output["action"].value_counts())

Final validation passed.
Ranked rows: 321546
Top-20 rows: 20

Reason-code counts:
reason_code
limited_visibility         241224
high_visibility             40162
high_visibility_low_ctr     40160
Name: count, dtype: int64

Action counts:
action
Lower-priority review    241224
Review next               40162
Review first              40160
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.